# Study 09 · 01 · Reproduction check

Nothing new is analysed until the regenerated models reproduce the paper. `code/run_fits.py historia_geo b3_obs` refits the paper's two primary arms (fixed hyperparameters, 5 seeds × 20 blocks). Before fitting, it runs the study-08 parity check: the fitting path equals the archived one on blocks 2008 and 2025.

**Pass criterion (protocol):** the regenerated six-season probabilities, passed through the paper's own `reproduce.py` functions, give exactly the Table II and Table S16 rows printed in `main.tex` and `supplement.tex`. The maximum |Δp| against the paper's bundle is reported.

The notebook also re-checks the training and calibration windows of every block on the fitted arms (leakage audit, part 3).

In [1]:
import os
import sys
from pathlib import Path

os.environ.update({"DENGUEKAY_XGB_DEVICE": "cpu", "DENGUEKAY_XGB_NJOBS": "2", "OMP_NUM_THREADS": "2",
                   "OPENBLAS_NUM_THREADS": "1", "VECLIB_MAXIMUM_THREADS": "1", "MLFLOW_DISABLE_AGENT_HINT": "1"})
STUDY = Path.cwd().parent
sys.path.insert(0, str(STUDY / "code"))

import numpy as np
import pandas as pd

import common as c

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

import importlib.util
import json

import experiments as ex
import leakage as lk

spec = importlib.util.spec_from_file_location("reproduce", c.PAPER_BUNDLE / "reproduce.py")
reproduce = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reproduce)
paper_manifest = json.loads((c.PAPER_BUNDLE / "manifest.json").read_text())

## 1 · Regenerated arms and baselines

In [2]:
inputs, _ = c.load_inputs(validate=False)
arms = {name: c.read_arm(name, columns=ex.PRED_COLUMNS) for name in ("b3_obs", "historia_geo")}
support = c.read_arm("b3_obs", seeds=(17,), columns=c.archive.OBSERVATIONS)
for name, frame in arms.items():
    first = frame[frame.semilla.eq(17)][c.archive.OBSERVATIONS].reset_index(drop=True)
    pd.testing.assert_frame_equal(first, support.reset_index(drop=True))
base = c.baselines(support, inputs["etiquetas"])
base.to_csv(c.OUTPUT / "baselines.csv.gz", index=False)
print({name: len(frame) for name, frame in arms.items()}, "| baseline rows:", len(base))

{'b3_obs': 974025, 'historia_geo': 974025} | baseline rows: 194805


## 2 · The six-season bundle against the paper's

In [3]:
def bundle_from(arms, base, seasons=c.DECISION_SEASONS):
    keys = ["bloque", "horizonte", "ubigeo", "semana_inicio"]
    out = ex.seasons_only(base, seasons)[[*keys, "temporada", "origen_cierre", "brote", "persistencia_prob"]]
    for name, label in (("b3_obs", "b3"), ("historia_geo", "history_geography")):
        wide = arms[name].pivot_table(index=keys, columns="semilla", values="probabilidad_calibrada")
        wide.columns = [f"{label}_s{s}" for s in wide.columns]
        out = out.merge(wide.reset_index(), on=keys, how="left", validate="one_to_one")
    return out.rename(columns={"temporada": "season", "horizonte": "horizon", "ubigeo": "district",
                               "semana_inicio": "target_week", "origen_cierre": "origin_close",
                               "brote": "elevated_week", "persistencia_prob": "probabilistic_persistence"})

mine = bundle_from(arms, base)
paper = pd.read_csv(c.PAPER_BUNDLE / "predictions.csv.gz", dtype={"district": str},
                    parse_dates=["target_week", "origin_close"])
both = paper.merge(mine, on=["season", "horizon", "district", "target_week"], suffixes=("_paper", "_new"),
                   validate="one_to_one")
probability_columns = ["probabilistic_persistence", *[f"{a}_s{s}" for a in ("b3", "history_geography") for s in c.SEEDS]]
check = pd.DataFrame({"column": probability_columns,
                      "max_abs_diff": [np.abs(both[f"{col}_paper"] - both[f"{col}_new"]).max() for col in probability_columns]})
print("rows paper / regenerated / matched:", len(paper), len(mine), len(both))
print("labels equal:", both.elevated_week_paper.equals(both.elevated_week_new),
      "| origins equal:", both.origin_close_paper.equals(both.origin_close_new))
check

rows paper / regenerated / matched: 61230 61230 61230
labels equal: True | origins equal: True


,column,max_abs_diff
0,probabilistic_persistence,1.998401e-15
1,b3_s17,4.440892e-16
2,b3_s29,4.440892e-16
3,b3_s41,4.440892e-16
4,b3_s53,4.440892e-16
5,b3_s67,4.440892e-16
6,history_geography_s17,4.440892e-16
7,history_geography_s29,4.440892e-16
8,history_geography_s41,4.440892e-16
9,history_geography_s53,4.440892e-16


## 3 · Table II and Table S16 from the regenerated probabilities (paper's `reproduce.py` functions)

In [4]:
scores = {}
for (season, horizon), g in mine.groupby(["season", "horizon"]):
    y = g.elevated_week.to_numpy(float)
    pp = np.mean((g.probabilistic_persistence - y) ** 2)
    for seed in c.SEEDS:
        scores[int(season), int(horizon), seed] = (np.mean((g[f"b3_s{seed}"] - y) ** 2),
                                                   np.mean((g[f"history_geography_s{seed}"] - y) ** 2), pp)
results = reproduce.summarize(scores, paper_manifest)
table_ii = [reproduce.main_table_row(r) for r in results]
table_s16 = list(reproduce.seasonal_rows(results, paper_manifest["seasons"]))
main_lines = (c.PAPER_BUNDLE.parent / "main.tex").read_text().splitlines()
supp_lines = (c.PAPER_BUNDLE.parent / "supplement.tex").read_text().splitlines()
reproduced = {"Table II": all(r in main_lines for r in table_ii), "Table S16": all(r in supp_lines for r in table_s16),
              "+1.49% without 2026": f"+{results[-1]['climate_change_without_2026_pct']:.2f}\\%" in "\n".join(main_lines)}
print("\n".join(table_ii))
reproduced

2 wk & 0.04276 & 0.04273 & 0.04420 & $-0.14$ ([-0.89, +0.46]) & $+0.49$ ([-4.80, +6.26])\\
3 wk & 0.04979 & 0.04938 & 0.05201 & $+0.41$ ([-0.63, +1.31]) & $+0.26$ ([-7.11, +7.79])\\
4 wk & 0.05569 & 0.05488 & 0.05950 & $+0.91$ ([-0.53, +2.33]) & $+2.11$ ([-6.12, +9.96])\\


{'Table II': True, 'Table S16': True, '+1.49% without 2026': True}

## 4 · Training and calibration windows of every block (leakage audit, part 3)

In [5]:
windows = pd.concat([lk.calibration_windows(name, 17).assign(brazo=name) for name in ("b3_obs", "historia_geo")])
fits = pd.concat([pd.read_csv(c.ARMS_DIR / f"{name}_s{s}" / "ajustes.csv", parse_dates=["corte"]).assign(brazo=name, semilla=s)
                  for name in ("b3_obs", "historia_geo") for s in c.SEEDS])
first_test = arms["b3_obs"].groupby("bloque").origen_cierre.min()
fits["primer_origen_prueba"] = fits.bloque.map(first_test)
summary = {
    "calibration: last label <= cutoff": bool((pd.to_datetime(windows.ultima_etiqueta.dropna()) <= pd.to_datetime(windows.loc[windows.ultima_etiqueta.notna(), "corte"])).all()),
    "calibration: cutoff <= first test origin": bool((pd.to_datetime(windows.corte) <= pd.to_datetime(windows.primer_origen_prueba)).all()),
    "Platt refit max |diff|": float(windows[["platt_a_dif", "platt_b_dif"]].max().max()),
    "blend weight refit max |diff|": float(windows.w_dif.max()),
    "training cutoff <= first test origin (all blocks, seeds)": bool((fits.corte <= fits.primer_origen_prueba).all()),
}
summary

{'calibration: last label <= cutoff': True,
 'calibration: cutoff <= first test origin': True,
 'Platt refit max |diff|': 2.1010970741031088e-14,
 'blend weight refit max |diff|': 0.0,
 'training cutoff <= first test origin (all blocks, seeds)': True}

In [6]:
passed = reproduced["Table II"] and reproduced["Table S16"] and reproduced["+1.49% without 2026"]
leak_free = (summary["calibration: last label <= cutoff"] and summary["calibration: cutoff <= first test origin"]
             and summary["Platt refit max |diff|"] < 1e-6 and summary["blend weight refit max |diff|"] < 1e-9
             and summary["training cutoff <= first test origin (all blocks, seeds)"])
print("REPRODUCTION:", "PASSED" if passed else "FAILED", "| max |dp| vs paper bundle:", check.max_abs_diff.max())
print("BLOCK WINDOWS:", "no leakage" if leak_free else "PROBLEM")
assert passed and leak_free

REPRODUCTION: PASSED | max |dp| vs paper bundle: 1.9984014443252818e-15
BLOCK WINDOWS: no leakage
